# DataFrame to LaTeX Table Converter

This notebook provides a dedicated tool (`LatexTableFormatter`) to format DataFrames into publication-ready LaTeX tables, with specialized support for:
- Concatenating central values and uncertainties into LaTeX `$val \pm unc$` syntax.
- Automatic uncertainty-based rounding (PDG standard: 2 significant figures for uncertainty with aligned central value decimal precision).
- Fixed decimal precision rounding.
- Auto-escaping LaTeX underscores and customizable tabular wrappers (`caption`, `label`, `column_format`, `resizebox`).

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from typing import Optional
from src.core.Validation.LatexTableFormatter import LatexTableFormatter

# Functions

In [4]:
def concatenate_pm_columns(
    df: pd.DataFrame,
    val_col: str,
    unc_col: str,
    new_col: Optional[str] = None,
    precision: Optional[int] = None,
    sig_figs: int = 2,
    drop_original: bool = False,
) -> pd.DataFrame:
    r"""Combines two columns into one formatted as '$val \pm unc$'."""
    return LatexTableFormatter.concatenate_columns(
        df=df,
        val_col=val_col,
        unc_col=unc_col,
        new_col=new_col,
        precision=precision,
        sig_figs=sig_figs,
        drop_original=drop_original,
    )


def dataframe_to_latex(
    df: pd.DataFrame,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    column_format: Optional[str] = None,
    escape_underscores: bool = True,
    resize_box: bool = False,
    index: bool = False,
) -> str:
    """Converts a DataFrame directly into a LaTeX table snippet."""
    return LatexTableFormatter.to_latex(
        df=df,
        caption=caption,
        label=label,
        column_format=column_format,
        escape_underscores=escape_underscores,
        resize_box=resize_box,
        index=index,
    )


TABLE_PRESETS = {
    "quadrant_strategy_metrics": {
        "pairs": {
            "PD": ("PD", "PD_unc"),
            "PF": ("PF", "PF_unc"),
            "SP": ("SP", "SP_unc"),
        },
        "keep_cols": ["Strategy", "PD", "PD_unc", "PF", "PF_unc", "SP", "SP_unc"],
    },
    "variable_js_divergence": {
        "pairs": {
            "JS Distance ($\\sqrt{D_{\\text{JS}}}$)": ("JS Distance (sqrt(D_JS))", "JS Distance (sqrt(D_JS)) Uncertainty"),
        },
        "keep_cols": ["Feature", "Strategy", "JS Distance (sqrt(D_JS))", "JS Distance (sqrt(D_JS)) Uncertainty"],
    },
    "multi_model_mean_summary": {
        "pairs": {
            "PD (Signal Eff)": ("PD mean [%]", "PD std [%]"),
            "PF (Fake Rate)": ("PF mean [%]", "PF std [%]"),
            "SP": ("SP mean [%]", "SP std [%]"),
        },
        "keep_cols": ["method", "PD mean [%]", "PD std [%]", "PF mean [%]", "PF std [%]", "SP mean [%]", "SP std [%]"],
    },
}


def discover_tables(project_root: Path = PROJECT_ROOT) -> pd.DataFrame:
    """Discovers saved CSV and Parquet tables under Plots/*/Tables and notebooks/Tables."""
    candidates = sorted(
        list((project_root / "Plots").glob("*/Tables/*.csv"))
        + list((project_root / "notebooks" / "Tables").glob("*.csv"))
        + list((project_root / "notebooks" / "Tables").glob("*.parquet"))
    )
    rows = [
        {
            "Index": i,
            "Folder": p.parent.parent.name if p.parent.name == "Tables" else p.parent.name,
            "File": p.name,
            "Path": str(p),
        }
        for i, p in enumerate(candidates)
    ]
    return pd.DataFrame(rows)


def load_table_df(table_path: Path) -> pd.DataFrame:
    """Loads a CSV or Parquet table into a DataFrame."""
    table_path = Path(table_path)
    if table_path.suffix.lower() == ".parquet":
        return pd.read_parquet(table_path)
    return pd.read_csv(table_path)


def resolve_latex_output_path(table_path: Path, custom_output_dir: Optional[Path] = None) -> Path:
    """Builds the standardized LaTeX output path: <run_dir>/LaTeX/<table_stem>_latex_table.tex."""
    table_path = Path(table_path)
    if custom_output_dir is not None:
        out_dir = Path(custom_output_dir)
    elif table_path.parent.name == "Tables":
        out_dir = table_path.parent.parent / "LaTeX"
    else:
        out_dir = table_path.parent / "LaTeX"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"{table_path.stem}_latex_table.tex"


def format_and_save_latex_table(
    table_path: Path,
    metric_pairs: Optional[dict] = None,
    keep_cols: Optional[list[str]] = None,
    precision: Optional[int] = None,
    sig_figs: int = 2,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    column_format: Optional[str] = None,
    resize_box: bool = False,
    custom_output_dir: Optional[Path] = None,
    max_rows: Optional[int] = None,
) -> tuple[pd.DataFrame, str, Path]:
    """Loads a table, applies preset or custom uncertainty formatting, and saves <stem>_latex_table.tex."""
    table_path = Path(table_path)
    df = load_table_df(table_path)
    if max_rows is not None:
        df = df.head(max_rows)

    matched_preset = next((v for k, v in TABLE_PRESETS.items() if k in table_path.stem), None)
    active_pairs = metric_pairs if metric_pairs is not None else (matched_preset["pairs"] if matched_preset else {})
    active_cols = keep_cols if keep_cols is not None else (matched_preset["keep_cols"] if matched_preset else None)

    if active_cols is not None:
        valid_cols = [c for c in active_cols if c in df.columns]
        if valid_cols:
            df = df[valid_cols]

    valid_pairs = {
        new_col: (v_col, u_col)
        for new_col, (v_col, u_col) in active_pairs.items()
        if v_col in df.columns and u_col in df.columns
    }
    if valid_pairs:
        formatted_df = LatexTableFormatter.merge_uncertainty_pairs(
            df,
            pairs=valid_pairs,
            precision=precision,
            sig_figs=sig_figs,
            drop_original=True,
        )
    else:
        formatted_df = df.copy()

    latex_str = dataframe_to_latex(
        formatted_df,
        caption=caption,
        label=label or f"tab:{table_path.stem}",
        column_format=column_format,
        resize_box=resize_box,
    )
    tex_path = resolve_latex_output_path(table_path, custom_output_dir=custom_output_dir)
    tex_path.write_text(latex_str, encoding="utf-8")
    return formatted_df, latex_str, tex_path

# Load, Format & Save Tables

In [5]:
# 1. Discover all available tables under Plots/*/Tables and notebooks/Tables
tables_catalog = discover_tables(PROJECT_ROOT)
display(tables_catalog)

,Index,Folder,File,Path
0,0,ConsolidatedAnalysis,.sys.v#.consolidated_pileup_fit_parameters.csv,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
1,1,ConsolidatedAnalysis,.sys.v#.consolidated_quadrant_counts.csv,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
2,2,ConsolidatedAnalysis,.sys.v#.consolidated_quadrant_strategy_metrics...,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
3,3,ConsolidatedAnalysis,.sys.v#.consolidated_variable_js_distance.csv,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
4,4,ConsolidatedAnalysis,consolidated_pileup_fit_parameters.csv,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
5,5,ConsolidatedAnalysis,consolidated_quadrant_counts.csv,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
6,6,ConsolidatedAnalysis,consolidated_quadrant_strategy_metrics.csv,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
7,7,ConsolidatedAnalysis,consolidated_variable_js_distance.csv,/eos/home-i04/p/pmourafr/egamma_master/Plots/C...
8,8,MultiModelOperatingPoints_athena_loose,.sys.v#.multi_model_operating_points_athena_lo...,/eos/home-i04/p/pmourafr/egamma_master/Plots/M...
9,9,MultiModelOperatingPoints_athena_loose,multi_model_operating_points_athena_loose_mean...,/eos/home-i04/p/pmourafr/egamma_master/Plots/M...


In [7]:
# 2. Select a table by Index from `tables_catalog` OR specify `CUSTOM_TABLE_PATH` directly
TABLE_INDEX: int = 0
CUSTOM_TABLE_PATH: Optional[Path] = PROJECT_ROOT / "Plots/MultiModelOperatingPoints_athena_loose/Tables/multi_model_operating_points_athena_loose_mean_summary.csv"

PRECISION: Optional[int] = None   # None = automatic PDG rounding, or int (e.g., 3)
SIG_FIGS: int = 2
CAPTION: Optional[str] = None
LABEL: Optional[str] = None       # None = auto tab:<file_stem>
MAX_ROWS: Optional[int] = None

selected_path = Path(CUSTOM_TABLE_PATH) if CUSTOM_TABLE_PATH is not None else Path(tables_catalog.loc[TABLE_INDEX, "Path"])
formatted_df, latex_str, saved_tex_path = format_and_save_latex_table(
    table_path=selected_path,
    precision=PRECISION,
    sig_figs=SIG_FIGS,
    caption=CAPTION,
    label=LABEL,
    max_rows=MAX_ROWS,
)

print(f"Source table: {selected_path}")
print(f"Saved LaTeX:  {saved_tex_path}")
display(formatted_df)
print("\n--- Exported LaTeX ---")
print(latex_str)

Source table: /eos/home-i04/p/pmourafr/egamma_master/Plots/MultiModelOperatingPoints_athena_loose/Tables/multi_model_operating_points_athena_loose_mean_summary.csv
Saved LaTeX:  /eos/home-i04/p/pmourafr/egamma_master/Plots/MultiModelOperatingPoints_athena_loose/LaTeX/multi_model_operating_points_athena_loose_mean_summary_latex_table.tex


,config_name,method,PD mean [%],PD std [%],SP mean [%],SP std [%],PF mean [%],PF std [%]
0,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [Best SP],81.36,7.34,83.20,6.46,14.90,6.04
1,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [Hard Cut],63.90,23.82,75.92,13.68,8.25,6.81
2,ModelV9_HighBatch_100Rings,Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\...,64.02,23.78,76.02,13.61,8.22,6.79
3,Athena,Athena Loose Cut,64.58,22.78,66.22,9.25,29.07,14.58
4,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [Best SP],81.54,6.51,82.79,6.41,15.91,7.02
5,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [Hard Cut],63.90,23.82,75.67,13.60,8.78,7.07
6,ModelV9_HighBatch_50Rings,Model V9 (50 Rings) [$y_\mathrm{cut}(\langle\m...,64.15,23.94,75.78,13.67,8.85,7.07
7,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [Best SP],81.58,7.29,82.56,6.77,16.41,6.99
8,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [Hard Cut],63.90,23.82,75.57,13.56,8.99,7.21
9,ModelV10_HighBatch_100Rings,Model V10 (100 Rings) [$y_\mathrm{cut}(\langle...,64.17,23.73,75.74,13.46,9.03,7.14



--- Exported LaTeX ---
\begin{table}[htbp]
\centering
\label{tab:multi_model_operating_points_athena_loose_mean_summary}
\begin{tabular}{l l c c c c c c}
\toprule
config\_name & method & PD mean [%] & PD std [%] & SP mean [%] & SP std [%] & PF mean [%] & PF std [%] \\
\midrule
ModelV9\_HighBatch\_100Rings & Model V9 (100 Rings) [Best SP] & 81.360000 & 7.340000 & 83.200000 & 6.460000 & 14.900000 & 6.040000 \\
ModelV9\_HighBatch\_100Rings & Model V9 (100 Rings) [Hard Cut] & 63.900000 & 23.820000 & 75.920000 & 13.680000 & 8.250000 & 6.810000 \\
ModelV9\_HighBatch\_100Rings & Model V9 (100 Rings) [$y_\mathrm{cut}(\langle\mu\rangle)$] & 64.020000 & 23.780000 & 76.020000 & 13.610000 & 8.220000 & 6.790000 \\
Athena & Athena Loose Cut & 64.580000 & 22.780000 & 66.220000 & 9.250000 & 29.070000 & 14.580000 \\
ModelV9\_HighBatch\_50Rings & Model V9 (50 Rings) [Best SP] & 81.540000 & 6.510000 & 82.790000 & 6.410000 & 15.910000 & 7.020000 \\
ModelV9\_HighBatch\_50Rings & Model V9 (50 Rings) [Hard 